# How Computers Compute $\ln x$: Range Reduction, Series, the AGM

Every call to `math.log` returns sixteen correct digits in a few nanoseconds — for $x = 10^{-300}$ as well as for $x = 10^{300}$. No table could be that large and no slide rule that precise. This notebook builds a working $\ln x$ from scratch with the same three ideas real math libraries use: **range reduction**, a **fast series** on a tiny interval, and precomputed constants. It ends with the arithmetic–geometric mean, used when thousands of digits are needed.

This is notebook 03 of module 03 (**Computing logarithms**) in the **Learning Logarithms** series.

## 1. Introduction

Notebook 01/04 gives the Mercator series

$$
\ln(1 + f) = f - \frac{f^2}{2} + \frac{f^3}{3} - \frac{f^4}{4} + \cdots, \qquad -1 < f \le 1 .
$$

So why not just use it? Try $\ln 10$: then $f = 9$ and the series **diverges**. Try $\ln 2$ ($f = 1$): the series converges, but the error after $N$ terms is about $1/(2N)$, so sixteen digits would need around $5 \times 10^{15}$ terms. Even $\ln 1.5$ ($f = 0.5$) needs about 50 terms. And a library must handle every positive float, from $5 \times 10^{-324}$ to $1.8 \times 10^{308}$.

The solution has the same shape as Briggs's (notebook 01): **first move the problem close to 1, then use a short linear-ish formula.** Briggs moved numbers towards $1$ with square roots; a computer does it almost for free, because floats are already stored as

$$
x = m \cdot 2^{k} \qquad\Longrightarrow\qquad \ln x = k \ln 2 + \ln m .
$$

One precomputed constant ($\ln 2$), one integer multiplication, and a logarithm of a number $m$ that is always between $0.71$ and $1.41$.

**Prerequisites**

- Change of base and binary logarithms — [Change of base](../00_foundations/04_change_of_base_and_common_bases.ipynb).
- The series for $\ln(1+x)$ and its faster variants — [Series for the logarithm](../01_the_natural_logarithm/04_series_for_the_logarithm.ipynb).
- Briggs's "move close to 1 first" idea — [Napier and Briggs tables](01_napier_and_briggs_tables.ipynb).
- Taylor polynomials and higher derivatives — [Higher-order derivatives](../../../LearningCalculus/hypersklill/02_derivaties/09_higher_order_derivatives.ipynb) (LearningCalculus).

After completing this notebook you will be able to:

- Decompose any float as $m \cdot 2^k$ with `math.frexp` and reduce $\ln x$ to $\ln m$ on a small interval.
- Derive the series $\ln m = 2\operatorname{artanh} s = 2(s + s^3/3 + s^5/5 + \cdots)$ with $s = (m-1)/(m+1)$, and explain why it converges fast.
- Write a complete `my_ln` in a dozen lines and measure its error in **units in the last place** (ulps) against `math.log`.
- Describe the arithmetic–geometric mean method for very high precision.
- Compute $\lfloor \log_2 n \rfloor$ exactly for huge integers with `int.bit_length()`, and see why `math.log2` can be wrong there.

## 2. Theory

### 2.1 Range reduction

Every positive float is stored as a **mantissa** times a power of two (notebook 04 shows the bits). Python exposes the split as `math.frexp(x)`, which returns $(m, k)$ with

$$
x = m \cdot 2^{k}, \qquad \tfrac12 \le m < 1 .
$$

Taking $\ln$ of both sides — exponent form on the left, log form on the right:

| Exponent form | Log form |
|---|---|
| $x = m \cdot 2^{k}$ | $\ln x = \ln m + k \ln 2$ |
| $10 = 0.625 \cdot 2^{4}$ | $\ln 10 = \ln 0.625 + 4\ln 2$ |

It is better to centre the interval on $1$: if $m < \sqrt{1/2}$, double $m$ and decrease $k$ by one. Then

$$
\sqrt{\tfrac12} \le m < \sqrt{2}, \qquad -0.347 \le \ln m < 0.347 .
$$

The whole positive axis, $2^{-1074}$ to $2^{1024}$, has been folded onto an interval of width less than a factor of two. Only $\ln m$ still needs work.

### 2.2 A series that converges fast: $\operatorname{artanh}$

Subtract the Mercator series for $\ln(1 - s)$ from that for $\ln(1 + s)$: the even powers cancel and the odd ones double,

$$
\ln\frac{1 + s}{1 - s} = 2\left(s + \frac{s^3}{3} + \frac{s^5}{5} + \frac{s^7}{7} + \cdots\right) = 2\operatorname{artanh} s .
$$

Given $m$, solve $\frac{1+s}{1-s} = m$ for $s$:

$$
s = \frac{m - 1}{m + 1}, \qquad \ln m = 2\left(s + \frac{s^3}{3} + \frac{s^5}{5} + \cdots\right).
$$

Two things make this fast:

- $s$ is much smaller than $m - 1$: for $m \in [\sqrt{1/2}, \sqrt2)$ we get $|s| \le \frac{\sqrt2 - 1}{\sqrt2 + 1} = 3 - 2\sqrt2 \approx 0.1716$;
- only odd powers appear, so each term is smaller than the previous one by a factor of at least $s^2 \approx 0.0294$ — about $1.5$ decimal digits per term.

Ten terms give double precision (about $16$ digits) on the whole interval. The same trick computes the constant itself: $\ln 2 = 2\operatorname{artanh}(1/3)$, since $\frac{1 + 1/3}{1 - 1/3} = 2$.

### 2.3 What real libraries do

Production libraries refine this, but the skeleton is the same. The classic **fdlibm** library (written at Sun, and still the reference for Java's `StrictMath.log`) reduces $x$ to $m \in [\sqrt2/2, \sqrt2)$, forms exactly this $s = (m-1)/(m+1)$, and replaces the tail of the artanh series by a short polynomial whose coefficients are tuned (a *minimax* fit) to minimise the maximum error on that interval. Many modern libraries add a small table of precomputed logarithms to shrink the interval even further, so the polynomial can be shorter. The constant $\ln 2$ is stored to more than double precision (split into a "high" and a "low" part) so that $k \ln 2$ adds no visible rounding error.

### 2.4 Measuring accuracy: ulps

Floats near $y$ are spaced $\operatorname{ulp}(y) \approx |y| \cdot 2^{-52}$ apart (one **u**nit in the **l**ast **p**lace; `np.spacing(y)`). The best any function can do is return the float nearest to the true value: an error of at most $0.5$ ulp. Good libraries guarantee less than $1$ ulp. We will measure `my_ln` in ulps against `math.log`.

### 2.5 The arithmetic–geometric mean (for many digits)

For thousands of digits, even a fast series is too slow: each term gains a fixed number of digits. Gauss's **arithmetic–geometric mean** gains a factor instead. Start from $a_0 = a$, $b_0 = b$ and iterate

$$
a_{n+1} = \frac{a_n + b_n}{2}, \qquad b_{n+1} = \sqrt{a_n b_n} .
$$

Both sequences converge to a common limit $\operatorname{AGM}(a, b)$, and the gap $a_n - b_n$ is roughly **squared** at each step: the number of correct digits doubles. For large $s$,

$$
\ln s \approx \frac{\pi}{2\,\operatorname{AGM}(1, 4/s)}, \qquad \text{with error of order } \frac{\ln s}{s^2} .
$$

To use it for any $x$, scale up first: pick $M$ with $s = x \cdot 2^{M}$ huge and compute $\ln x = \ln s - M \ln 2$. This is how arbitrary-precision libraries compute logarithms to millions of digits. For ordinary floats it is not faster than the series, so we show it only briefly.

### 2.6 Integers: $\log_2$ by counting bits

For a positive integer $n$, `n.bit_length()` is the number of binary digits, $b$, so $2^{b-1} \le n < 2^{b}$ and

$$
\lfloor \log_2 n \rfloor = b - 1 .
$$

This is **exact** for integers of any size. `math.log2(n)` first rounds $n$ to the nearest float, which for $n = 2^{60} - 1$ is exactly $2^{60}$ — so `math.floor(math.log2(n))` returns $60$, while the right answer is $59$.

## 3. Worked Examples

### Example 1 — $\ln 10$ by range reduction

`math.frexp(10)` gives $10 = 0.625 \cdot 2^4$. Since $0.625 < 0.7071$, re-centre: $10 = 1.25 \cdot 2^3$. Then

$$
\ln 10 = 3 \ln 2 + \ln 1.25 = 3(0.693147) + \ln 1.25 .
$$

### Example 2 — $\ln 1.25$ by the artanh series

$s = \frac{1.25 - 1}{1.25 + 1} = \frac{0.25}{2.25} = \frac19$. Then

$$
\ln 1.25 = 2\left(\frac19 + \frac{1}{3 \cdot 9^3} + \frac{1}{5 \cdot 9^5} + \cdots\right) = 2(0.1111111 + 0.0004572 + 0.0000034 + \cdots) = 0.2231436 .
$$

Three terms already give seven digits ($\ln 1.25 = 0.22314355\ldots$). So $\ln 10 = 2.0794415 + 0.2231436 = 2.3025851$, correct to seven digits.

### Example 3 — compare with Mercator

Mercator for $\ln 1.25$ uses $f = 0.25$: terms shrink by a factor of $0.25$ each, and the signs alternate. Three terms give $0.25 - 0.03125 + 0.0052083 = 0.2239583$ — only two correct digits. The artanh version shrinks by $s^2 = 1/81$ per term.

### Example 4 — a tiny number

$x = 10^{-300}$: `math.frexp` gives $k = -996$, $m \approx 0.67$, so $\ln x \approx -996 \ln 2 + \ln 0.67 \approx -690.37 - 0.40 = -690.78$, and indeed $-300 \ln 10 = -690.78$. The magnitude of $x$ is handled entirely by the integer $k$.

### Example 5 — bit length

$1000$ in binary is $1111101000$: ten bits. So $\lfloor \log_2 1000 \rfloor = 9$, and indeed $2^9 = 512 \le 1000 < 1024 = 2^{10}$.

## 4. Python Examples

### 4.1 Range reduction with `math.frexp`

In [ ]:
import math

SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


print("       x            frexp          centred m      k")
for x in [10, 0.1, 1.0, 1e-300, 1e300, 2**0.5, 5e-324]:
    centred_m, k = reduce_range(x)
    print(f"{x:>10.4g}   {str(math.frexp(x)):>28}   {centred_m:.6f}  {k:>6}")

Even the smallest subnormal float $5 \times 10^{-324} = 2^{-1074}$ reduces to $m = 1$, $k = -1074$.

### 4.2 The artanh series and the constant $\ln 2$

First the constant. We sum the series for $\ln 2 = 2\operatorname{artanh}(1/3)$ with 40-digit `Decimal` arithmetic and round the result to a float **once** — the way a library author precomputes constants.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40


def ln_two_series(terms):
    s = Decimal(1) / 3
    total = Decimal(0)
    for n in range(terms):
        total += s ** (2 * n + 1) / (2 * n + 1)
    return 2 * total


for terms in [5, 10, 20, 40]:
    print(f"{terms:>2} terms: {ln_two_series(terms)}")
print("decimal ln(2): ", Decimal(2).ln())
print("as a float equals math.log(2):", float(ln_two_series(40)) == math.log(2))

Each term adds about one digit ($\log_{10} 9 \approx 0.95$), so 40 terms give the 40-digit constant.

### 4.3 A complete `my_ln`

Range reduction, the artanh series on $|s| \le 0.1716$ (evaluated with Horner's rule, writing $2s + 2s \cdot s^2 (\tfrac13 + \tfrac{s^2}{5} + \cdots)$ so the big term is added last), and the constant.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m


for x in [10, 0.1, 2, 1e-300, 1e300, 3.14159, 1.0000001]:
    print(f"x = {x!r:<10}  my_ln = {my_ln(x): .16f}   math.log = {math.log(x): .16f}")

### 4.4 Error in ulps over a million-fold range

Test $100{,}000$ random $x$ spread log-uniformly from $e^{-50}$ to $e^{50}$, and measure $|$`my_ln(x)` $-$ `math.log(x)`$|$ in units of `np.spacing(math.log(x))`.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m

import numpy as np

rng = np.random.default_rng(seed=0)
test_points = np.exp(rng.uniform(-50, 50, size=100_000))
ulp_errors = np.array([abs(my_ln(x) - math.log(x)) / np.spacing(abs(math.log(x))) for x in test_points])

print("largest error:            ", ulp_errors.max(), "ulps")
print("identical to math.log:    ", f"{np.mean(ulp_errors == 0):.1%}")
print("within 1 ulp of math.log: ", f"{np.mean(ulp_errors <= 1):.3%}")

Twelve lines of Python agree with the C library bit-for-bit on most inputs and are never more than 2 ulps away. The remaining differences come from rounding inside our own arithmetic — `math.log` is more careful with $\ln 2$ (split into two parts) and with the final addition.

### 4.5 $\log_{10}$ and $\log_2$ from $\ln$

Any other base is one division away (change of base). For $\log_2$ the range reduction is even simpler, because $\log_2 2^k = k$ exactly: $\log_2 x = k + \ln m / \ln 2$.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m


LN10 = float(Decimal(10).ln())


def my_log10(x):
    return my_ln(x) / LN10


def my_log2(x):
    m, k = reduce_range(x)
    return k + my_ln(m) / LN2


for x in [1000, 2**20, 0.001, 7]:
    print(f"x = {x:<8}  log10: {my_log10(x)!r:<20} math.log10: {math.log10(x)!r:<20} log2: {my_log2(x)!r:<20} math.log2: {math.log2(x)!r}")
print()
print("math.log(1000, 10) =", repr(math.log(1000, 10)))

Note the classic surprise: `my_log10(1000)` gives $2.9999999999999996$, and so does `math.log(1000, 10)` — the division by $\ln 10$ adds a rounding. `math.log10` uses a dedicated routine and returns exactly $3$. Always compare with a tolerance.

### 4.6 The AGM method, briefly

Using `Decimal` with 60 digits: compute $\pi$ with the Gauss–Legendre algorithm (itself an AGM), then $\ln 2$ from $\ln 2^{M} \approx \pi / (2\operatorname{AGM}(1, 4/2^{M}))$ with $M = 100$. Watch the gap $a_n - b_n$: the number of zeros roughly doubles each step.

In [ ]:
from decimal import Decimal, getcontext

getcontext().prec = 60


def agm(a, b, show=False):
    for step in range(1, 40):
        a, b = (a + b) / 2, (a * b).sqrt()
        if show:
            print(f"  step {step:>2}: a - b = {abs(a - b):.2e}")
        if abs(a - b) < Decimal(10) ** -55:
            break
    return a


def gauss_legendre_pi():
    a, b, t, p = Decimal(1), 1 / Decimal(2).sqrt(), Decimal(1) / 4, Decimal(1)
    for _ in range(8):
        a_next = (a + b) / 2
        b = (a * b).sqrt()
        t -= p * (a - a_next) ** 2
        a, p = a_next, 2 * p
    return (a + b) ** 2 / (4 * t)


pi = gauss_legendre_pi()
M = 100
print("AGM(1, 4 / 2^100):")
agm_value = agm(Decimal(1), Decimal(4) / Decimal(2) ** M, show=True)
ln_two_agm = pi / (2 * agm_value) / M

print("ln 2 by AGM :", str(ln_two_agm)[:52])
print("decimal ln 2:", str(Decimal(2).ln())[:52])
print("difference  :", f"{abs(ln_two_agm - Decimal(2).ln()):.1e}")

The first steps only halve the gap, while $a$ and $b$ are still far apart. Once the gap is small it is squared on every step: $2 \times 10^{-5}$, $2 \times 10^{-9}$, $3 \times 10^{-17}$, $4 \times 10^{-33}$, $10^{-61}$. Eleven steps give the full sixty digits. A series would need a number of terms proportional to the number of digits; the AGM needs a number of steps proportional to its **logarithm**.

### 4.7 Integers: `bit_length` versus `math.log2`

In [ ]:
import math

for n in [1000, 2**60 - 1, 2**60, 3**200]:
    exact_floor = n.bit_length() - 1
    float_floor = math.floor(math.log2(n))
    print(f"bit_length - 1 = {exact_floor:>4}   floor(math.log2(n)) = {float_floor:>4}   n = {str(n)[:20]}{'...' if len(str(n)) > 20 else ''}")

print()
print("math.log works on integers beyond the float range:", math.log(10**400), " (400 ln 10 =", 400 * math.log(10), ")")

For $n = 2^{60} - 1$ the float rounds up to $2^{60}$ and `math.log2` answers $60.0$, one too many. For integers too large to become floats at all, such as $10^{400}$, CPython's `math.log` itself uses range reduction: it splits the integer as $m \cdot 2^{k}$ and returns $\ln m + k \ln 2$ — exactly section 2.1.

## 5. Visualizations

### 5.1 Range reduction: a staircase plus a small wiggle

On a logarithmic $x$-axis, $\ln x$ is a straight line. Range reduction splits it into a staircase $k \ln 2$ (integer steps, one per factor of two) and a small sawtooth $\ln m$ that never leaves $[-0.35, 0.35]$. Only the sawtooth needs the series.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m

import matplotlib.pyplot as plt
import numpy as np

x_values = np.geomspace(2**-4, 2**4, 2000)
reduced = [reduce_range(x) for x in x_values]
k_values = np.array([k for m, k in reduced])
ln_m_values = np.array([math.log(m) for m, k in reduced])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_values, np.log(x_values), color="black", linewidth=2, label=r"$\ln x$")
ax.step(x_values, k_values * LN2, where="post", color="tab:blue", label=r"staircase $k \ln 2$")
ax.plot(x_values, ln_m_values, color="tab:orange", label=r"sawtooth $\ln m$, $m \in [\sqrt{1/2}, \sqrt{2})$")
ax.axhspan(-0.347, 0.347, color="tab:orange", alpha=0.1)
ax.set_xscale("log", base=2)
ax.set_xlabel("x (log scale, base 2)")
ax.set_ylabel("value")
ax.set_title(r"$\ln x = k \ln 2 + \ln m$")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper left")
plt.show()

### 5.2 Why the artanh series and the centred interval

Error after $N$ terms at the worst point of each interval, on a log axis: Mercator on $m \in [0.5, 1)$ (worst $f = -0.5$), artanh on $[0.5, 1)$ (worst $|s| = 1/3$), and artanh on the centred $[\sqrt{1/2}, \sqrt 2)$ (worst $|s| = 0.1716$). The horizontal line is double precision.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from decimal import Decimal, getcontext

getcontext().prec = 50
term_counts = np.arange(1, 61)


def mercator_error(f, terms):
    partial = sum(Decimal(-1) ** (n + 1) * Decimal(f) ** n / n for n in range(1, terms + 1))
    return float(abs(partial - (1 + Decimal(f)).ln()))


def artanh_error(s, terms):
    partial = 2 * sum(Decimal(s) ** (2 * n + 1) / (2 * n + 1) for n in range(terms))
    m = (1 + Decimal(s)) / (1 - Decimal(s))
    return float(abs(partial - m.ln()))


worst_centred_s = 3 - 2 * Decimal(2).sqrt()
curves = {
    "Mercator, m in [0.5, 1)": [mercator_error(-0.5, n) for n in term_counts],
    "artanh, m in [0.5, 1)": [artanh_error(-Decimal(1) / 3, n) for n in term_counts],
    "artanh, m in [0.707, 1.414)": [artanh_error(worst_centred_s, n) for n in term_counts],
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, errors in curves.items():
    ax.semilogy(term_counts, errors, "o-", markersize=3, label=label)
ax.axhline(2.0**-53, color="black", linestyle="--", label="double precision (1.1e-16)")
ax.set_xlabel("number of terms")
ax.set_ylabel("worst-case error (log scale)")
ax.set_ylim(1e-30, 10)          # below this the 50-digit reference itself runs out of digits
ax.set_title("Fewer terms: smaller s, odd powers only")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

for label, errors in curves.items():
    needed = next(n for n, e in zip(term_counts, errors) if e < 2.0**-53)
    print(f"{label:<30} reaches 1.1e-16 after {needed} terms")

### 5.3 Where the ulp errors fall

The error of `my_ln` against `math.log`, in ulps, over the test range. It is zero for most inputs and never exceeds two ulps; there is no region where it grows.

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(seed=0)
test_points = np.exp(rng.uniform(-50, 50, size=20_000))
signed_ulp_errors = np.array([(my_ln(x) - math.log(x)) / np.spacing(abs(math.log(x))) for x in test_points])

fig, (ax_scatter, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})
ax_scatter.scatter(test_points, signed_ulp_errors, s=3, alpha=0.3)
ax_scatter.set_xscale("log")
ax_scatter.set_xlabel("x (log scale)")
ax_scatter.set_ylabel("my_ln - math.log (ulps)")
ax_scatter.set_title("Error across 43 orders of magnitude")
ax_scatter.grid(True, which="both", alpha=0.3)

values, counts = np.unique(np.round(signed_ulp_errors), return_counts=True)
ax_hist.bar(values, counts, color="tab:green")
ax_hist.set_yscale("log")
ax_hist.set_xlabel("error (ulps)")
ax_hist.set_ylabel("count (log scale)")
ax_hist.set_title("Distribution")
ax_hist.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Use `math.frexp` to write $x = 100$ as $m \cdot 2^k$ with $m \in [\sqrt{1/2}, \sqrt 2)$, and express $\ln 100$ through $\ln 2$ and $\ln m$.
2. Compute $\ln 1.5$ with the first **two** terms of the artanh series by hand. What is $s$? How large is the error?
3. What is `(1000).bit_length()`? What is `(1024).bit_length()`? Give $\lfloor \log_2 n \rfloor$ for both.

### Intermediate

4. **Estimate it first, then compute.** With $|s| \le 0.17$ and each term about $s^2 \approx 0.03$ times the previous, estimate how many artanh terms give $16$ correct digits. Then check with the error curve or with code.
5. How many terms of $\ln 2 = 2\operatorname{artanh}(1/3)$ are needed to get $\ln 2$ to double precision ($10^{-16}$)? Compare with the Mercator series for $\ln 2$ ($f = 1$), whose error after $N$ terms is about $1/(2N)$.
6. Write `my_log2` (section 4.5) without using `my_ln` for the integer part, and test that `my_log2(2**k)` is exactly `k` for $k = -1000, \dots, 1000$.

### Advanced

7. Explain why centring the interval at $1$ (i.e. using $[\sqrt{1/2}, \sqrt 2)$ rather than $[1/2, 1)$) matters, by comparing the largest $|s|$ and the number of terms needed in each case.
8. Use the AGM formula with **floats**: $\ln x \approx \pi / (2\operatorname{AGM}(1, 4/s)) - M\ln 2$ with $s = x \cdot 2^{M}$. Try $M = 10, 20, 30, 40$ for $x = 10$. Why does the error first fall and then stop improving?
9. Write `decimal_digits(n)` that returns the number of decimal digits of a positive integer $n$ **exactly**, using `bit_length` for a first estimate $\lfloor (b - 1)\log_{10} 2 \rfloor + 1$ and one comparison with a power of ten to correct it. Test against `len(str(n))` on random integers up to $10^{1000}$.

## 7. Solutions

1. `math.frexp(100)` $= (0.78125, 7)$; $0.78125 \ge 0.7071$, so $100 = 0.78125 \cdot 2^7$ and $\ln 100 = 7\ln 2 + \ln 0.78125 = 4.85203 - 0.24686 = 4.60517$.
2. $s = 0.5/2.5 = 0.2$; $\ln 1.5 \approx 2(0.2 + 0.008/3) = 2(0.2026667) = 0.4053333$. True: $0.4054651$; error $1.3 \times 10^{-4}$, close to the next term $2s^5/5 = 1.3 \times 10^{-4}$.
3. $1000 = 1111101000_2$ has $10$ bits, so $\lfloor\log_2 1000\rfloor = 9$. $1024 = 10000000000_2$ has $11$ bits, so $\lfloor\log_2 1024\rfloor = 10$ — exactly $\log_2 1024$.
4. Each term gains $\log_{10}(1/0.03) \approx 1.5$ digits and the first term already gives about $1$, so about $10$ terms. The code finds $10$ terms at the worst point $s = 0.1716$ (section 5.2 counts the same).
5. Each artanh term gains a factor $9$ ($\approx 0.95$ digits), so about $16$ terms; the code finds $16$. Mercator would need about $N = 1/(2 \times 10^{-16}) = 5 \times 10^{15}$ terms.
6. `my_log2(x) = k + ln(m)/ln 2` with $m$ from `reduce_range`. For $x = 2^k$, `frexp` gives $m = 0.5$, re-centred to $m = 1$, $\ln 1 = 0$, so the result is the integer $k$ exactly (check below).
7. On $[1/2, 1)$ the worst point is $m = 1/2$, $s = -1/3$, $s^2 = 1/9$: about $16$ terms. On $[\sqrt{1/2}, \sqrt2)$ the worst $|s| = 0.1716$, $s^2 = 0.0294$: $10$ terms. Centring at $1$ saves six terms, more than a third of the work.
8. The formula's own error is of order $\ln s / s^2$, which falls fast as $M$ grows ($M = 10$: error $3 \times 10^{-7}$; $M = 20$: $5 \times 10^{-13}$). But $\ln s \approx M\ln 2$ grows, and subtracting $M\ln 2$ from it cancels digits: float rounding of relative size $10^{-16}$ in $\ln s \approx 25$ leaves an error floor of a few times $10^{-15}$ ($M = 30$ and $M = 40$ both give $2.7 \times 10^{-15}$).
9. With $b$ = `n.bit_length()`, $2^{b-1} \le n < 2^b$, so the digit count is either $d = \lfloor (b-1)\log_{10} 2 \rfloor + 1$ or $d + 1$; check `n >= 10**d` to decide. All random tests agree with `len(str(n))`.

Numerical checks:

In [ ]:
import math
from decimal import Decimal, getcontext

getcontext().prec = 40
LN2 = float(Decimal(2).ln())          # precomputed once, to 40 digits, then rounded to a float
SQRT_HALF = math.sqrt(0.5)


def reduce_range(x):
    # x = m * 2**k with sqrt(1/2) <= m < sqrt(2)
    m, k = math.frexp(x)              # 0.5 <= m < 1
    if m < SQRT_HALF:
        m, k = 2 * m, k - 1
    return m, k


def artanh_tail(s_squared, terms):
    # 1/3 + s^2/5 + s^4/7 + ... evaluated by Horner's rule
    total = 0.0
    for n in reversed(range(1, terms)):
        total = total * s_squared + 1 / (2 * n + 1)
    return total


def my_ln(x, terms=11):
    if x <= 0:
        raise ValueError("ln needs x > 0")
    m, k = reduce_range(x)
    s = (m - 1) / (m + 1)
    s_squared = s * s
    ln_m = 2 * s + 2 * s * s_squared * artanh_tail(s_squared, terms)
    return k * LN2 + ln_m

import random

m, k = reduce_range(100)
print("Ex 1:", math.frexp(100), "->", m, k, " 7 ln2 + ln m =", round(k * math.log(2) + math.log(m), 5), " ln 100 =", round(math.log(100), 5))

s = 0.5 / 2.5
two_terms = 2 * (s + s**3 / 3)
print(f"Ex 2: s = {s}, two terms = {two_terms:.7f}, ln 1.5 = {math.log(1.5):.7f}, error {abs(two_terms - math.log(1.5)):.1e}")
print("Ex 3:", (1000).bit_length(), (1024).bit_length(), "->", (1000).bit_length() - 1, (1024).bit_length() - 1)

getcontext().prec = 50


def artanh_partial(s, terms):
    s = Decimal(s)
    return 2 * sum(s ** (2 * n + 1) / (2 * n + 1) for n in range(terms))


worst_s = 3 - 2 * Decimal(2).sqrt()
exact = ((1 + worst_s) / (1 - worst_s)).ln()
print("Ex 4: terms needed at s = 0.1716:", next(n for n in range(1, 40) if abs(artanh_partial(worst_s, n) - exact) < Decimal(2) ** -53))
print("Ex 5: terms for ln 2 = 2 artanh(1/3):", next(n for n in range(1, 40) if abs(artanh_partial(Decimal(1) / 3, n) - Decimal(2).ln()) < Decimal(2) ** -53))


def my_log2(x):
    m, k = reduce_range(x)
    return k + my_ln(m) / LN2


print("Ex 6: my_log2(2**k) == k for all k in -1000..1000:", all(my_log2(2.0**k) == k for k in range(-1000, 1001)))
print("Ex 7: terms needed at s = -1/3:", next(n for n in range(1, 40) if abs(artanh_partial(-Decimal(1) / 3, n) - Decimal("0.5").ln()) < Decimal(2) ** -53))


def float_agm(a, b):
    for _ in range(30):
        a, b = (a + b) / 2, math.sqrt(a * b)
    return a


for M in [10, 20, 30, 40]:
    s = 10 * 2.0**M
    estimate = math.pi / (2 * float_agm(1.0, 4 / s)) - M * math.log(2)
    print(f"Ex 8: M = {M}: error {abs(estimate - math.log(10)):.1e}")


def decimal_digits(n):
    estimate = math.floor((n.bit_length() - 1) * math.log10(2)) + 1
    return estimate + 1 if n >= 10**estimate else estimate


random.seed(4)
tests = [random.randrange(1, 10 ** random.randint(1, 1000)) for _ in range(2000)] + [10**j for j in range(300)] + [10**j - 1 for j in range(1, 300)]
print("Ex 9: all agree with len(str(n)):", all(decimal_digits(n) == len(str(n)) for n in tests))

## 8. Conclusion

- **Range reduction** is the decisive step: $x = m \cdot 2^k$ (`math.frexp`) gives $\ln x = k\ln 2 + \ln m$, folding every positive float onto $m \in [\sqrt{1/2}, \sqrt2)$ where $|\ln m| < 0.35$. It is Briggs's "move close to 1 first", done by reading the exponent bits.
- On that interval the **artanh series** $\ln m = 2(s + s^3/3 + s^5/5 + \cdots)$ with $s = (m-1)/(m+1)$, $|s| \le 0.1716$, converges at $1.5$ digits per term: about ten terms for double precision. Libraries use the same $s$ with a tuned polynomial, plus constants like $\ln 2$ precomputed to extra precision.
- Accuracy is measured in **ulps**. Our twelve-line `my_ln` is within $2$ ulps of `math.log` everywhere tested and identical on most inputs.
- For thousands of digits, the **AGM** gives $\ln s \approx \pi / (2\operatorname{AGM}(1, 4/s))$ with quadratic convergence: the digits double each step.
- For integers, `n.bit_length() - 1` $= \lfloor \log_2 n \rfloor$ exactly; `math.log2` can be off by one once $n$ exceeds $2^{53}$.

**Next:** notebook 04 looks at the float format itself — a logarithmic number line — and at what goes wrong (underflow, cancellation) and how `log1p`, `expm1` and `logsumexp` fix it.